# 键盘采集示教数据

在给定环境中采集示教数据。
任务是抓起杯子并放到盘子上。当杯子在盘子上、夹爪打开且末端执行器位于杯子上方时，环境会判定成功。

<img src="./media/teleop.gif" width="480" height="360">

键位说明：WASD 控制 x-y 平面移动，R/F 控制 z 轴，Q/E 控制倾斜，方向键控制其余旋转。

空格键切换夹爪开合状态，Z 键会重置环境并丢弃当前回合缓存数据。

叠加图像说明：
- 右上：Agent 视角
- 右下：腕部（第一人称）视角
- 左上：侧视图
- 左下：俯视图


注意采集数据时，尽量避免像视频中那样操作，需更精准地采集。

In [ ]:
# pip install -r requirements.txt
# 推荐python310环境

In [1]:
import os, importlib
os.environ["IK_BACKEND"] = "pybullet"
os.environ["XARM7_CONTACT_STOP_CLOSE"] = "1"
os.environ["GRIPPER_RATE_PER_STEP"] = "0.02"   # 还想更稳可再小一点

In [2]:
import sys
import random
import numpy as np
import os
from PIL import Image
from mujoco_env.y_env import SimpleEnv
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

In [3]:
# If you want to randomize the object positions, set this to None
# If you fix the seed, the object positions will be the same every time
SEED = 0 
# SEED = None <- Uncomment this line to randomize the object positions

REPO_NAME = 'datawhale_eai_pnp_xarm7' #pick-and-place
NUM_DEMO = 1 # Number of demonstrations to collect
ROOT = "./demo_data_xarm7" # The root directory to save the demonstrations

In [4]:
# import os, importlib
# os.environ["IK_BACKEND"] = "pybullet"
# os.environ["XARM7_CONTACT_STOP_CLOSE"] = "1"
# os.environ["GRIPPER_RATE_PER_STEP"] = "0.02"   # 还想更稳可再小一点

# import mujoco_env.y_env as y_env
# importlib.reload(y_env)
# from mujoco_env.y_env import SimpleEnv


In [5]:
import env_config

[Env Info] 运行环境已就绪，当前工作目录: C:\Users\kewei\lerobot-mujoco-tutorial


记得解压这个文件夹

![fig1](./media/fig1.png)

In [6]:
TASK_NAME = 'Put mug cup on the plate' 
xml_path = './asset/example_scene_xarm7.xml'
# xml_path = 'asset/example_scene_y.xml'
# Define the environment
PnPEnv = SimpleEnv(xml_path, seed = SEED, state_type = 'joint_angle')


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[27] n_qvel:[25] n_qacc:[25] n_ctrl:[8]
 integrator:[IMPLICITFAST]

n_body:[25]
 [0/25] [world] mass:[0.00]kg
 [1/25] [front_object_table] mass:[1.00]kg
 [2/25] [camera] mass:[0.00]kg
 [3/25] [camera2] mass:[0.00]kg
 [4/25] [camera3] mass:[0.00]kg
 [5/25] [link_base] mass:[0.89]kg
 [6/25] [link1] mass:[2.38]kg
 [7/25] [link2] mass:[1.87]kg
 [8/25] [link3] mass:[1.64]kg
 [9/25] [link4] mass:[1.73]kg
 [10/25] [link5] mass:[1.32]kg
 [11/25] [link6] mass:[1.32]kg
 [12/25] [link7] mass:[0.17]kg
 [13/25] [xarm_gripper_base_link] mass:[0.54]kg
 [14/25] [left_outer_knuckle] mass:[0.03]kg
 [15/25] [left_finger] mass:[0.05]kg
 [16/25] [left_inner_knuckle] mass:[0.02]kg
 [17/25] [right_outer_knuckle] mass:[0.03]kg
 [18/25] [right_finger] mass:[0.05]kg
 [19/25] [right_inner_knuckle] mass:[0.02]kg
 [20/25] [camera_wrist] mass:[0.00]kg
 [21/25] [body_obj_mug_5] mass:[0.00]kg
 [2

## 定义数据集特征并创建你的数据集
数据集结构如下：
```
fps = 20,
features={
    "observation.image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.wrist_image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channel"],
    },
    "observation.state": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["state"], # x, y, z, roll, pitch, yaw
    },
    "action": {
        "dtype": "float32",
        "shape": (8,),
        "names": ["action"], # 7 joint angles and 1 gripper
    },
    "obj_init": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["obj_init"], # just the initial position of the object. Not used in training.
    },
},
```

数据将保存在 `./demo_data` 目录中。


In [7]:
create_new = True
if os.path.exists(ROOT):
    print(f"Directory {ROOT} already exists.")
    ans = input("Do you want to delete it? (y/n) ")
    if ans == 'y':
        import shutil
        shutil.rmtree(ROOT)
    else:
        create_new = False


if create_new:
    dataset = LeRobotDataset.create(
                repo_id=REPO_NAME,
                root = ROOT, 
                robot_type="xarm7",
                fps=20, # 20 frames per second
                features={
                    "observation.image": {
                        "dtype": "image",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channels"],
                    },
                    "observation.wrist_image": {
                        "dtype": "image",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channel"],
                    },
                    "observation.state": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["state"], # x, y, z, roll, pitch, yaw
                    },
                    "action": {
                        "dtype": "float32",
                        "shape": (8,),
                        "names": ["action"], # 7 joint angles and 1 gripper
                    },
                    "obj_init": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["obj_init"], # just the initial position of the object. Not used in training.
                    },
                },
                image_writer_threads=2,
                image_writer_processes=0,
        )
else:
    print("Load from previous dataset")
    dataset = LeRobotDataset(REPO_NAME, root=ROOT)

Directory ./demo_data_xarm7 already exists.


Do you want to delete it? (y/n)  y


## 键盘控制
你可以用键盘遥操作机械臂并采集数据
```
---------     -----------------------
   w       ->        backward
s  a  d        left   forward   right
---------      -----------------------
In x, y plane

---------
R: Moving Up
F: Moving Down
---------
In z axis

---------
Q: Tilt left
E: Tilt right
UP: Look Upward
Down: Look Donward
Right: Turn right
Left: Turn left
---------
For rotation

---------
SPACEBAR: Toggle Gripper
--------

---------
z: reset
--------
```
重置环境会删除当前示教回合的缓存数据，并重新开始采集。


关于 Rotation（旋转）部分的详细解释：
在机器人控制中，这部分通常对应 3D 空间中的 欧拉角（Euler Angles）：

Tilt (Q/E): 对应 Roll（翻滚角）。想象机械臂末端像拧螺丝一样左右倾斜。

Look Upward/Downward (UP/Down): 对应 Pitch（俯仰角）。控制机械臂末端的“头”抬起或低下。

Turn Left/Right (Left/Right): 对应 Yaw（偏航角）。控制机械臂末端在水平方向上左右摆动。

### 现在开始遥操作并采集数据

**要收到成功信号，你需要松开夹爪并将末端执行器上移到杯子上方。**


杯子和盘子中心在 XY 上要足够近（< 0.1）

夹爪要“真正打开”（rh_r1 < 0.1，这个很容易卡住）

末端执行器（tcp_link）要抬高到 z > 0.9

以上都满足后，才会触发 done=True

In [8]:
action = np.zeros(7)
episode_id = 0
record_flag = False # Start recording when the robot starts moving
while PnPEnv.env.is_viewer_alive() and episode_id < NUM_DEMO:
    PnPEnv.step_env()
    if PnPEnv.env.loop_every(HZ=20):
        # check if the episode is done
        done = PnPEnv.check_success()
        if done: 
            # Save the episode data and reset the environment
            dataset.save_episode()
            PnPEnv.reset(seed = SEED)
            episode_id += 1
        # Teleoperate the robot and get delta end-effector pose with gripper
        action, reset  = PnPEnv.teleop_robot()
        if not record_flag and sum(action) != 0:
            record_flag = True
            print("Start recording")
        if reset:
            # Reset the environment and clear the episode buffer
            # This can be done by pressing 'z' key
            PnPEnv.reset(seed=SEED)
            # PnPEnv.reset()
            dataset.clear_episode_buffer()
            record_flag = False
        # Step the environment
        # Get the end-effector pose and images
        ee_pose = PnPEnv.get_ee_pose()
        agent_image,wrist_image = PnPEnv.grab_image()
        # # resize to 256x256
        agent_image = Image.fromarray(agent_image)
        wrist_image = Image.fromarray(wrist_image)
        agent_image = agent_image.resize((256, 256))
        wrist_image = wrist_image.resize((256, 256))
        agent_image = np.array(agent_image)
        wrist_image = np.array(wrist_image)
        joint_q = PnPEnv.step(action)
        if record_flag:
            # Add the frame to the dataset
            dataset.add_frame( {
                    "observation.image": agent_image,
                    "observation.wrist_image": wrist_image,
                    "observation.state": ee_pose, 
                    "action": joint_q,
                    "obj_init": PnPEnv.obj_init_pose,
                    # "task": TASK_NAME,
                }, task = TASK_NAME
            )
        PnPEnv.render(teleop=True)

Start recording


Map:   0%|          | 0/827 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

DONE INITIALIZATION


这里相当于只用采集一条数据

In [9]:
PnPEnv.env.close_viewer()

In [10]:
# Clean up the images folder
import shutil
shutil.rmtree(dataset.root / 'images')